# Mission抽選シミュレーション解析

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MicomProcedure/change-change-change/blob/main/Tools/Analysis/mission_draw_analysis.ipynb)

Unity Editorの `CCC/Mission Draw Simulation` が `Logs/CCC/Simulation/` に出力した1回分のCSV（`catalog` / `draws` / `candidates`）を読み込み、次の観点で解析します。

1. **Mission単位の選出割合**: 特定のMissionが出過ぎていないか、狙ったMissionが出ていないか
2. **Kind・Tierの出現分布**: Section（Tier解放の進行）ごとの推移
3. **パラメータ・適合率の質**: Sectionごとの目標式適合割合の範囲、選出Missionの適合率と抽選ごとの目標値の差、Pool内順位の偏り
4. **連続出題の偏り**: Session内で同じKind・Missionが続いていないか（`SequentialFlow` の実行のみ）

## 使い方

1. `Logs/CCC/Simulation/` の `{SessionId}_catalog.csv`・`_draws.csv`・`_candidates.csv` の3ファイルを、Google Driveの `SIMULATION_DIRECTORY`（既定: `マイドライブ/CCC/Simulation`）へアップロードします。
2. 下の設定セルで `SESSION_ID` を指定します。空文字のときはフォルダ内で最も新しい実行を使います。
3. 「ランタイム → すべてのセルを実行」で実行します。初回はDriveへのアクセス許可を求められます。

グラフはカーソルを重ねると個々のMissionの値を確認できます。グラフ内の表記は英語（Kind名などの識別子）です。

## 設定

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

IS_COLAB = "google.colab" in sys.modules
DRIVE_MOUNT_POINT = "/content/drive"
DEFAULT_SIMULATION_DIRECTORY = f"{DRIVE_MOUNT_POINT}/MyDrive/CCC/Simulation"

# 環境変数で上書きできます（ローカルのJupyterで実行する場合など）。
SIMULATION_DIRECTORY = Path(os.environ.get("CCC_SIMULATION_DIRECTORY", DEFAULT_SIMULATION_DIRECTORY))
# 解析するSessionId（例: "sim_20261019_141110_seed0"）。空文字ならフォルダ内で最も新しい実行を使います。
SESSION_ID = os.environ.get("CCC_SIMULATION_SESSION_ID", "")

TOP_MISSION_COUNT = 20          # 選出回数・偏りの上位として表示するMission数
CONCENTRATION_TOP_FRACTION = 0.1  # 「上位何割のMissionが全選出の何割を占めるか」の上位割合
SECTION_BIN_WIDTH = 5           # Section推移のグラフで束ねるSection数
REPEAT_WINDOW_DRAWS = 5         # 同じMissionの再出題を数える直前の抽選数

pd.set_option("display.max_rows", None)  # 未選出Missionなどの一覧を省略せずに表示する
pd.set_option("display.width", 200)

In [ ]:
if IS_COLAB and str(SIMULATION_DIRECTORY).startswith(DRIVE_MOUNT_POINT):
    from google.colab import drive

    drive.mount(DRIVE_MOUNT_POINT)

## グラフ設定

色は役割ごとに固定します。極性（Positive / Negated）は分類色、Tierは1色の明度段階、ヒートマップは1色の連続階調、目標値・基準値は無彩色の線です。

In [ ]:
POLARITY_ORDER = ["Positive", "Negated"]
POLARITY_COLORS = {"Positive": "#2a78d6", "Negated": "#eb6834"}
SERIES_COLOR = "#2a78d6"
REFERENCE_COLOR = "#52514e"
MISSING_CELL_COLOR = "#d9d8d3"
ORDINAL_BLUE_STEPS = ["#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQUENTIAL_BLUE_SCALE = ["#f5f9fe", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

PLOT_TEMPLATE = "simple_white"
PLOT_HEIGHT = 420


def tier_colors(tiers):
    # Tier数に応じて明度段階から等間隔に色を取り、Tier番号→色の対応を返す。
    ordered_tiers = sorted(tiers)
    if len(ordered_tiers) == 1:
        return {ordered_tiers[0]: ORDINAL_BLUE_STEPS[len(ORDINAL_BLUE_STEPS) // 2]}
    step_indices = np.linspace(0, len(ORDINAL_BLUE_STEPS) - 1, len(ordered_tiers)).round().astype(int)
    return {tier: ORDINAL_BLUE_STEPS[index] for tier, index in zip(ordered_tiers, step_indices)}


def style_figure(figure, title, height=PLOT_HEIGHT):
    figure.update_layout(template=PLOT_TEMPLATE, title=title, height=height, hovermode="closest", bargap=0.15)
    figure.update_xaxes(showgrid=False)
    figure.update_yaxes(showgrid=True, gridcolor="#e6e5e0")
    return figure

## 読み込みと検証

3表を読み込み、列の過不足、`draws` の選出候補がすべて `catalog` に存在すること、各抽選の解放済み候補数が `catalog` から再計算した値と一致することを確認します。不一致は別の実行のCSVを混ぜた可能性を示すため、例外で停止します。

In [ ]:
TABLE_SUFFIXES = ("catalog", "draws", "candidates")
DRAWS_FILE_PATTERN = "*_draws.csv"
DRAWS_FILE_SUFFIX = "_draws.csv"

REQUIRED_COLUMNS = {
    "catalog": ["SessionId", "CatalogIndex", "Tier", "Kind", "ParameterA", "ParameterB", "IsNegated", "MatchCount", "EvaluatedCount", "Ratio"],
    "draws": [
        "SessionId", "ConfigDisplayName", "TargetExpressionMatchRatio", "TargetExpressionMatchRatioReductionPerSection",
        "MinimumTargetExpressionMatchRatio", "MissionParameterSelectionPoolSize", "NegatedMissionSelectionWeight",
        "DrawIndex", "Section", "IsInitialDraw", "CurrentTier", "EnabledCount", "UnlockedCount",
        "SelectedExpressionGroup", "SelectedTier", "SelectedPolarity", "SelectedKind", "KindStageCount",
        "TargetRatioLowerBound", "TargetRatioUpperBound", "TargetRatio", "PoolSize", "SelectedRank",
        "SelectedParameterA", "SelectedParameterB", "SelectedRatio", "SelectedDifference",
    ],
    "candidates": ["SessionId", "DrawIndex", "Rank", "Tier", "Kind", "ParameterA", "ParameterB", "IsNegated", "Ratio", "IsInPool", "IsSelected"],
}


def find_latest_session_id(directory):
    # フォルダ内のdraws表のうち更新日時が最も新しいもののSessionIdを返す。
    draws_paths = sorted(directory.glob(DRAWS_FILE_PATTERN), key=lambda path: path.stat().st_mtime)
    if not draws_paths:
        raise FileNotFoundError(f"{directory} に {DRAWS_FILE_PATTERN} が見つかりません。")
    return draws_paths[-1].name[: -len(DRAWS_FILE_SUFFIX)]


def resolve_table_paths(directory, session_id):
    # SessionIdに対応する3表のパスを返す。欠けている表があれば例外にする。
    paths = {suffix: directory / f"{session_id}_{suffix}.csv" for suffix in TABLE_SUFFIXES}
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("次のCSVが見つかりません: " + ", ".join(missing))
    return paths


def read_tables(paths):
    # 3表を読み込み、必須列の欠落を検証する。
    tables = {suffix: pd.read_csv(path) for suffix, path in paths.items()}
    for suffix, table in tables.items():
        missing_columns = sorted(set(REQUIRED_COLUMNS[suffix]) - set(table.columns))
        if missing_columns:
            raise ValueError(f"{suffix} 表に必須列がありません: {missing_columns}")
    return tables


resolved_session_id = SESSION_ID or find_latest_session_id(SIMULATION_DIRECTORY)
raw_tables = read_tables(resolve_table_paths(SIMULATION_DIRECTORY, resolved_session_id))
print(f"SessionId: {resolved_session_id}")
for suffix, table in raw_tables.items():
    print(f"  {suffix:<10} {len(table):>7} rows")

In [ ]:
PARAMETER_COUNT_BY_KIND = {
    "Even": 0, "Odd": 0, "Integer": 0, "Decimal": 0, "Positive": 0, "Negative": 0, "Prime": 0, "Zero": 0,
    "Square": 0, "Composite": 0, "AtLeast": 1, "AtMost": 1, "Exact": 1, "Multiple": 1, "Absolute": 1,
    "OnesDigit": 1, "DecimalFirstDigit": 1, "Between": 2, "Modulo": 2, "Divisor": 1, "Power": 1,
    "BetweenZeroAndOne": 0, "TwoDigit": 0, "TensDigit": 1, "DigitSum": 1, "RepeatingDecimal": 0,
}
NEGATED_LABEL_PREFIX = "NOT "
POSITIVE_POLARITY = "Positive"
NEGATED_POLARITY = "Negated"
MISSION_KEY_COLUMNS = ["Kind", "ParameterA", "ParameterB", "IsNegated"]


def format_parameters(kind, parameter_a, parameter_b):
    # Kindが使うパラメータだけを `8` / `3,5` 形式で返す。パラメータを持たないKindは空文字。未知のKindは両パラメータを表示する。
    parameter_count = PARAMETER_COUNT_BY_KIND.get(kind, 2)
    return ",".join(str(parameter) for parameter in [parameter_a, parameter_b][:parameter_count])


def format_mission_label(kind, parameter_a, parameter_b, is_negated):
    # Kindとパラメータから `AtLeast(8)` / `NOT Between(3,5)` 形式の構造ラベルを作る。
    parameters = format_parameters(kind, parameter_a, parameter_b)
    body = f"{kind}({parameters})" if parameters else kind
    return NEGATED_LABEL_PREFIX + body if is_negated else body


def to_polarity(is_negated):
    return NEGATED_POLARITY if is_negated else POSITIVE_POLARITY


def read_run_config(draws):
    # draws表の先頭行から、1回の実行で共通の抽選パラメータを取り出す。
    first_row = draws.iloc[0]
    return {
        "ConfigDisplayName": first_row["ConfigDisplayName"],
        "TargetExpressionMatchRatio": float(first_row["TargetExpressionMatchRatio"]),
        "TargetExpressionMatchRatioReductionPerSection": float(first_row["TargetExpressionMatchRatioReductionPerSection"]),
        "MinimumTargetExpressionMatchRatio": float(first_row["MinimumTargetExpressionMatchRatio"]),
        "PoolSize": int(first_row["MissionParameterSelectionPoolSize"]),
        "NegatedMissionSelectionWeight": float(first_row["NegatedMissionSelectionWeight"]),
    }


def prepare_catalog(catalog, run_config):
    # Catalogに構造ラベル、極性、抽選対象かどうか、目標式適合割合の上限（Section 1の値）との差、全候補が残った場合のPool内順位を付ける。
    prepared = catalog.copy()
    prepared["IsNegated"] = prepared["IsNegated"].astype(bool)
    prepared["Polarity"] = prepared["IsNegated"].map(to_polarity)
    prepared["MissionLabel"] = [
        format_mission_label(kind, a, b, negated)
        for kind, a, b, negated in prepared[["Kind", "ParameterA", "ParameterB", "IsNegated"]].itertuples(index=False)
    ]
    prepared["IsEnabled"] = ~prepared["IsNegated"] | (run_config["NegatedMissionSelectionWeight"] > 0)
    prepared["Difference"] = (prepared["Ratio"] - run_config["TargetExpressionMatchRatio"]).abs()
    # Strategyと同じく差の昇順・Catalog順で並べた、同じTier・極性・Kind内の順位。抽選ごとの目標値はSectionで下がるため、上限に対する静的な目安とする。
    prepared = prepared.sort_values(["Difference", "CatalogIndex"])
    prepared["StaticPoolRank"] = prepared.groupby(["Tier", "IsNegated", "Kind"]).cumcount() + 1
    return prepared.sort_values("CatalogIndex").reset_index(drop=True)


def prepare_draws(draws, catalog):
    # 抽選行に極性・Session番号・構造ラベルを付け、選出候補をCatalogと結合する。
    prepared = draws.copy()
    prepared["IsInitialDraw"] = prepared["IsInitialDraw"].astype(bool)
    prepared["IsNegated"] = prepared["SelectedPolarity"] == NEGATED_POLARITY
    prepared["SessionNumber"] = prepared["IsInitialDraw"].cumsum()
    selected_keys = prepared.rename(columns={
        "SelectedKind": "Kind", "SelectedParameterA": "ParameterA", "SelectedParameterB": "ParameterB",
    })
    merged = selected_keys.merge(
        catalog[MISSION_KEY_COLUMNS + ["CatalogIndex", "Tier", "MissionLabel", "StaticPoolRank"]],
        on=MISSION_KEY_COLUMNS,
        how="left",
        validate="many_to_one",
    )
    unmatched = merged[merged["CatalogIndex"].isna()]
    if not unmatched.empty:
        raise ValueError(f"catalog に存在しない選出候補が {len(unmatched)} 件あります（DrawIndex={unmatched['DrawIndex'].head().tolist()}...）。")
    merged["CatalogIndex"] = merged["CatalogIndex"].astype(int)
    return merged.rename(columns={"Kind": "SelectedKind", "ParameterA": "SelectedParameterA", "ParameterB": "SelectedParameterB", "Tier": "CatalogTier"})


def validate_unlocked_counts(draws, catalog):
    # 各抽選のEnabledCount・UnlockedCountがCatalogから再計算した値と一致することを確認する。
    enabled = catalog[catalog["IsEnabled"]]
    expected_unlocked = draws["CurrentTier"].map(lambda tier: int((enabled["Tier"] <= tier).sum()))
    mismatched = draws[(draws["UnlockedCount"] != expected_unlocked) | (draws["EnabledCount"] != len(enabled))]
    if not mismatched.empty:
        raise ValueError(f"catalog と draws の候補数が {len(mismatched)} 行で一致しません。別の実行のCSVが混ざっていないか確認してください。")


def has_draw_sequences(draws):
    # 先頭がSession初回抽選で、Session内でSectionが1ずつ進む（SequentialFlowの）実行かどうか。
    if not draws["IsInitialDraw"].iloc[0]:
        return False
    section_steps = draws.groupby("SessionNumber")["Section"].diff().dropna()
    return not section_steps.empty and bool((section_steps == 1).all())


run_config = read_run_config(raw_tables["draws"])
catalog = prepare_catalog(raw_tables["catalog"], run_config)
draws = prepare_draws(raw_tables["draws"], catalog)
candidates = raw_tables["candidates"]
validate_unlocked_counts(draws, catalog)
IS_SEQUENTIAL_RUN = has_draw_sequences(draws)
print("検証OK")

## 実行概要

In [ ]:
overview = pd.Series({
    "SessionId": resolved_session_id,
    "Config": run_config["ConfigDisplayName"],
    "Draws": len(draws),
    "Sessions (initial draws)": int(draws["IsInitialDraw"].sum()),
    "Sections": f"{draws['Section'].min()} - {draws['Section'].max()}",
    "Sequential run": IS_SEQUENTIAL_RUN,
    "Catalog candidates": len(catalog),
    "Enabled candidates": int(catalog["IsEnabled"].sum()),
    "TargetExpressionMatchRatio (upper bound)": run_config["TargetExpressionMatchRatio"],
    "TargetExpressionMatchRatioReductionPerSection": run_config["TargetExpressionMatchRatioReductionPerSection"],
    "MinimumTargetExpressionMatchRatio": run_config["MinimumTargetExpressionMatchRatio"],
    "PoolSize": run_config["PoolSize"],
    "NegatedMissionSelectionWeight": run_config["NegatedMissionSelectionWeight"],
})
display(overview.to_frame("Value"))
display(Markdown("**Catalog候補数（Tier × 極性）**"))
display(pd.crosstab(catalog["Tier"], catalog["Polarity"], margins=True, margins_name="Total"))

## 1. Mission単位の選出割合

Mission（Kind・パラメータ・極性の組）ごとに選出回数を数えます。抽選ごとに解放済みかどうかが変わるため、次の2つの基準と比べます。

| 列 | 意味 |
| --- | --- |
| `ExposedDraws` | そのMissionのTierが解放済みだった抽選回数（出題され得た回数） |
| `UniformExpected` | 各抽選で解放済み候補から一様に選んだ場合の期待選出回数（Σ 1/UnlockedCount）。**抽選アルゴリズムの狙いではなく、偏りの大きさを測る物差し** |
| `Lift` | `Selected / UniformExpected`。1より大きいほど一様抽選より出やすい |
| `StaticPoolRank` | 同じTier・極性・Kindの全候補が残った場合の、目標式適合割合の上限 `TargetExpressionMatchRatio`（Section 1の値）との差による順位。`PoolSize` より大きいMissionは、式グループや直前除外で上位候補が外れたとき、またはSectionが進んで目標値が下がったときしか選ばれない |

抽選はKindを一様に選んでからKind内の候補を選ぶため、候補の少ないKind（`Even` など）の各Missionは構造的に `Lift` が大きくなります。

In [ ]:
def count_draws_by_current_tier(draws):
    return draws.groupby("CurrentTier").agg(DrawCount=("DrawIndex", "size"), UnlockedCount=("UnlockedCount", "first"))


def summarize_mission_frequency(draws, catalog):
    # Catalogの各Missionについて、選出回数・出題され得た抽選回数・一様抽選での期待回数を集計する。
    draws_by_tier = count_draws_by_current_tier(draws)
    selected_counts = draws["CatalogIndex"].value_counts()
    summary = catalog.copy()
    summary["Selected"] = summary["CatalogIndex"].map(selected_counts).fillna(0).astype(int)
    summary["Share"] = summary["Selected"] / len(draws)

    def exposure(tier):
        exposed_tiers = draws_by_tier[draws_by_tier.index >= tier]
        return int(exposed_tiers["DrawCount"].sum()), float((exposed_tiers["DrawCount"] / exposed_tiers["UnlockedCount"]).sum())

    exposures = summary["Tier"].map(exposure)
    summary["ExposedDraws"] = [draw_count for draw_count, _ in exposures]
    summary["UniformExpected"] = [expected for _, expected in exposures]
    summary.loc[~summary["IsEnabled"], ["ExposedDraws", "UniformExpected"]] = 0
    summary["Lift"] = np.where(summary["UniformExpected"] > 0, summary["Selected"] / summary["UniformExpected"].where(summary["UniformExpected"] > 0), np.nan)
    return summary


def calculate_concentration(selected_counts, top_fraction):
    # 選出回数の偏りを、上位割合のシェア・実効Mission数・ジニ係数で表す。
    counts = np.sort(np.asarray(selected_counts, dtype=float))[::-1]
    total = counts.sum()
    shares = counts / total
    top_count = max(1, int(np.ceil(len(counts) * top_fraction)))
    ascending = np.sort(counts)
    cumulative = np.cumsum(ascending)
    gini = 1 - 2 * np.sum(cumulative) / (len(counts) * total) + 1 / len(counts)
    return pd.Series({
        "Exposed missions": len(counts),
        "Selected at least once": int((counts > 0).sum()),
        f"Share of top {top_fraction:.0%} missions ({top_count})": shares[:top_count].sum(),
        "Effective mission count (1/Σp²)": 1 / np.sum(shares ** 2),
        "Gini coefficient": gini,
    })


MISSION_COLUMNS = ["MissionLabel", "Tier", "Polarity", "Ratio", "StaticPoolRank", "Selected", "Share", "ExposedDraws", "UniformExpected", "Lift"]

mission_frequency = summarize_mission_frequency(draws, catalog)
exposed_missions = mission_frequency[mission_frequency["ExposedDraws"] > 0]
display(calculate_concentration(exposed_missions["Selected"], CONCENTRATION_TOP_FRACTION).to_frame("Value"))

In [ ]:
def plot_rank_frequency(missions):
    # 選出回数の多い順に並べたMissionの選出回数と累積シェア（2つのグラフ）を描く。
    ordered = missions.sort_values("Selected", ascending=False).reset_index(drop=True)
    ordered["Order"] = ordered.index + 1
    ordered["CumulativeShare"] = ordered["Selected"].cumsum() / ordered["Selected"].sum()
    count_figure = px.bar(
        ordered, x="Order", y="Selected", color="Polarity", color_discrete_map=POLARITY_COLORS,
        category_orders={"Polarity": POLARITY_ORDER}, hover_data=["MissionLabel", "Tier", "Lift"],
        labels={"Order": "Mission (sorted by selections)", "Selected": "Selections"},
    )
    style_figure(count_figure, "Selections per mission (exposed missions, sorted)").show()
    cumulative_figure = px.line(
        ordered, x="Order", y="CumulativeShare", hover_data=["MissionLabel", "Selected"],
        labels={"Order": "Mission (sorted by selections)", "CumulativeShare": "Cumulative share of draws"},
    )
    cumulative_figure.update_traces(line=dict(color=SERIES_COLOR, width=2))
    cumulative_figure.update_yaxes(tickformat=".0%", range=[0, 1.02])
    style_figure(cumulative_figure, "Cumulative share of draws").show()


plot_rank_frequency(exposed_missions)

### 出過ぎているMission

選出回数の上位と、一様抽選に対する `Lift` の上位です。

In [ ]:
display(Markdown(f"**選出回数 上位{TOP_MISSION_COUNT}件**"))
display(exposed_missions.nlargest(TOP_MISSION_COUNT, "Selected")[MISSION_COLUMNS].round(4))
display(Markdown(f"**Lift 上位{TOP_MISSION_COUNT}件**"))
display(exposed_missions.nlargest(TOP_MISSION_COUNT, "Lift")[MISSION_COLUMNS].round(4))

### 出ていないMission

解放済みだったのに一度も選ばれなかったMissionと、Kind・Tier単位の網羅率です。`StaticPoolRank` が `PoolSize` を超えるMissionは、目標式適合割合の上限から遠いためPoolに入りにくいことが原因です。

In [ ]:
never_selected = exposed_missions[exposed_missions["Selected"] == 0]
display(Markdown(f"**未選出 {len(never_selected)} / {len(exposed_missions)} 件**（`StaticPoolRank > PoolSize` の件数: {int((never_selected['StaticPoolRank'] > run_config['PoolSize']).sum())}）"))
display(never_selected.sort_values(["Kind", "Polarity", "StaticPoolRank"])[MISSION_COLUMNS].round(4))


def summarize_coverage(missions, group_columns):
    # グループごとに、解放済みMission数・1回以上選ばれたMission数・網羅率・選出回数を集計する。
    coverage = missions.groupby(group_columns).agg(
        ExposedMissions=("CatalogIndex", "size"),
        SelectedMissions=("Selected", lambda selected: int((selected > 0).sum())),
        Selections=("Selected", "sum"),
    )
    coverage["CoverageRatio"] = coverage["SelectedMissions"] / coverage["ExposedMissions"]
    return coverage


display(Markdown("**Kind × 極性ごとの網羅率**"))
display(summarize_coverage(exposed_missions, ["Kind", "Polarity"]).round(3))
display(Markdown("**Tierごとの網羅率**"))
display(summarize_coverage(exposed_missions, ["Tier"]).round(3))

## 2. Kind・Tierの出現分布

In [ ]:
def plot_kind_counts(draws, catalog):
    # Kind別の選出回数を極性で積み上げ、Catalog候補数をホバーに添えて描く。
    catalog_counts = catalog[catalog["IsEnabled"]].groupby("Kind").size().rename("CatalogCandidates")
    kind_counts = draws.groupby(["SelectedKind", "SelectedPolarity"]).size().rename("Selections").reset_index()
    kind_counts = kind_counts.merge(catalog_counts, left_on="SelectedKind", right_index=True, how="left")
    kind_order = draws["SelectedKind"].value_counts().index.tolist()
    figure = px.bar(
        kind_counts, x="SelectedKind", y="Selections", color="SelectedPolarity", color_discrete_map=POLARITY_COLORS,
        category_orders={"SelectedKind": kind_order, "SelectedPolarity": POLARITY_ORDER}, hover_data=["CatalogCandidates"],
        labels={"SelectedKind": "Kind", "SelectedPolarity": "Polarity"},
    )
    style_figure(figure, "Selections per kind").show()


def add_section_bin(draws, bin_width):
    binned = draws.copy()
    bin_start = (binned["Section"] - 1) // bin_width * bin_width + 1
    binned["SectionBin"] = bin_start.astype(str) + "-" + (bin_start + bin_width - 1).astype(str)
    binned["SectionBinStart"] = bin_start
    return binned


def section_bin_order(binned):
    return binned.drop_duplicates("SectionBin").sort_values("SectionBinStart")["SectionBin"].tolist()


def plot_tier_share_by_section(draws, bin_width):
    # Sectionの区間ごとに、選ばれたMissionのTier構成比を積み上げで描く。
    binned = add_section_bin(draws, bin_width)
    shares = pd.crosstab(binned["SectionBin"], binned["SelectedTier"], normalize="index").stack().rename("Share").reset_index()
    colors = tier_colors(draws["SelectedTier"].unique())
    shares["SelectedTier"] = shares["SelectedTier"].astype(str)
    figure = px.bar(
        shares, x="SectionBin", y="Share", color="SelectedTier",
        color_discrete_map={str(tier): color for tier, color in colors.items()},
        category_orders={"SectionBin": section_bin_order(binned), "SelectedTier": [str(tier) for tier in sorted(colors)]},
        labels={"SectionBin": "Section", "SelectedTier": "Selected tier"},
    )
    figure.update_yaxes(tickformat=".0%")
    style_figure(figure, f"Selected tier share by section ({bin_width} sections per bar)").show()


def plot_kind_share_heatmap(draws, bin_width):
    # Sectionの区間 × Kindの選出割合をヒートマップで描く。
    binned = add_section_bin(draws, bin_width)
    shares = pd.crosstab(binned["SelectedKind"], binned["SectionBin"], normalize="columns")
    shares = shares[section_bin_order(binned)]
    shares = shares.loc[draws["SelectedKind"].value_counts().index]
    figure = px.imshow(
        shares, color_continuous_scale=SEQUENTIAL_BLUE_SCALE, aspect="auto",
        labels={"x": "Section", "y": "Kind", "color": "Share"},
    )
    figure.update_traces(hovertemplate="Section %{x}<br>%{y}: %{z:.1%}<extra></extra>")
    figure.update_coloraxes(colorbar_tickformat=".0%")
    style_figure(figure, "Kind share by section", height=max(PLOT_HEIGHT, 28 * len(shares))).show()


plot_kind_counts(draws, catalog)
plot_tier_share_by_section(draws, SECTION_BIN_WIDTH)
plot_kind_share_heatmap(draws, SECTION_BIN_WIDTH)

## 3. パラメータ・適合率の質

目標式適合割合は抽選ごとに、`TargetRatioLowerBound`〜`TargetRatioUpperBound` の範囲から選ばれます（`TargetRatio`）。範囲の上限は `TargetExpressionMatchRatio`、下限はSectionが進むごとに `TargetExpressionMatchRatioReductionPerSection` ずつ下がり、`MinimumTargetExpressionMatchRatio` で止まります。低下量が0の場合と `FullMissionDrawStrategy` の実行では、範囲の幅は0です。

ここでは、Sectionごとの範囲と実際の選出適合率、選出Missionの適合率（`SelectedRatio`）がその抽選の目標値からどれだけ離れているか、Pool内のどの順位が選ばれているか、Kind内のどのパラメータが選ばれているかを確認します。

In [ ]:
def plot_ratio_by_section(draws):
    # Sectionごとに、目標式適合割合の範囲（無彩色の破線）と選出Missionの適合率の分布（箱ひげ図）を描く。
    bounds = draws.groupby("Section")[["TargetRatioLowerBound", "TargetRatioUpperBound"]].first().reset_index()
    figure = px.box(
        draws, x="Section", y="SelectedRatio", points="outliers", hover_data=["MissionLabel", "TargetRatio"],
        labels={"SelectedRatio": "Expression match ratio of selected mission"},
    )
    figure.update_traces(marker_color=SERIES_COLOR, line_color=SERIES_COLOR, name="Selected ratio", showlegend=True)
    for column, label, dash in (
        ("TargetRatioUpperBound", "Target upper bound", "dash"),
        ("TargetRatioLowerBound", "Target lower bound", "dot"),
    ):
        figure.add_trace(go.Scatter(
            x=bounds["Section"], y=bounds[column], mode="lines", name=label,
            line=dict(color=REFERENCE_COLOR, dash=dash, width=2),
        ))
    style_figure(figure, "Selected match ratio and target range by section").show()


def plot_signed_target_gap(draws):
    # 選出Missionの適合率からその抽選の目標値を引いた差の分布を描く。正は目標より満たす式が多い（易しい側の）候補が選ばれたことを示す。
    gaps = draws.assign(TargetGap=draws["SelectedRatio"] - draws["TargetRatio"])
    figure = px.histogram(gaps, x="TargetGap", nbins=40, labels={"TargetGap": "Selected ratio - draw target ratio"})
    figure.update_traces(marker_color=SERIES_COLOR, marker_line_color="white", marker_line_width=1)
    figure.add_vline(x=0, line_color=REFERENCE_COLOR, line_dash="dash")
    figure.update_yaxes(title="Draws")
    style_figure(figure, "Gap between selected ratio and draw target").show()


def plot_difference_by_kind(draws):
    # Kindごとに、選出Missionの適合率とその抽選の目標値との差の分布を箱ひげ図で描く。
    kind_order = draws.groupby("SelectedKind")["SelectedDifference"].median().sort_values(ascending=False).index.tolist()
    figure = px.box(
        draws, x="SelectedKind", y="SelectedDifference", points="outliers", hover_data=["MissionLabel"],
        category_orders={"SelectedKind": kind_order}, labels={"SelectedKind": "Kind", "SelectedDifference": "|ratio - draw target|"},
    )
    figure.update_traces(marker_color=SERIES_COLOR, line_color=SERIES_COLOR)
    style_figure(figure, "Distance from target match ratio by kind").show()


plot_ratio_by_section(draws)
plot_signed_target_gap(draws)
plot_difference_by_kind(draws)

### Pool内順位

Pool内はランダム抽選のため、`PoolSize` が同じ抽選では `SelectedRank` がほぼ均等になるはずです。`PoolSize` が設定値より小さい抽選は、そのKindの残り候補が少なかったことを示します。

In [ ]:
display(Markdown("**PoolSize別の SelectedRank 構成比**"))
display(pd.crosstab(draws["PoolSize"], draws["SelectedRank"], normalize="index", margins=False).round(3).assign(Draws=draws["PoolSize"].value_counts()))

pool_shortage = draws.assign(IsPoolShort=draws["PoolSize"] < run_config["PoolSize"]).groupby("SelectedKind").agg(
    Draws=("DrawIndex", "size"), PoolShortDraws=("IsPoolShort", "sum"), MeanKindStageCount=("KindStageCount", "mean"),
)
pool_shortage["PoolShortRatio"] = pool_shortage["PoolShortDraws"] / pool_shortage["Draws"]
display(Markdown(f"**Kind別: Poolが設定値 {run_config['PoolSize']} 未満だった抽選の割合**"))
display(pool_shortage.sort_values("PoolShortRatio", ascending=False).round(3))

### Kind内のパラメータ分布

パラメータを持つKindについて、解放済みの全Mission（未選出を含む）の選出回数を並べます。1パラメータのKindは棒グラフ（ホバーで適合率と `StaticPoolRank` を確認可能）、2パラメータのKind（`Between`・`Modulo`）は極性ごとのヒートマップです。

In [ ]:
PARAMETER_AXIS_TITLES = {"Between": ("Lower bound", "Width"), "Modulo": ("Divisor", "Remainder")}


def plot_single_parameter_distribution(kind_missions, kind):
    # 1パラメータのKindについて、パラメータ値ごとの選出回数を極性別に並べて描く。
    figure = px.bar(
        kind_missions, x="ParameterA", y="Selected", color="Polarity", barmode="group",
        color_discrete_map=POLARITY_COLORS, category_orders={"Polarity": POLARITY_ORDER},
        hover_data=["MissionLabel", "Tier", "Ratio", "StaticPoolRank", "Lift"],
        labels={"ParameterA": "Parameter", "Selected": "Selections"},
    )
    figure.update_xaxes(dtick=1)
    style_figure(figure, f"{kind}: selections per parameter").show()


def plot_pair_parameter_distribution(kind_missions, kind):
    # 2パラメータのKindについて、極性ごとにA×Bの選出回数をヒートマップで描く。Catalogにない組は灰色の背景になる。
    a_title, b_title = PARAMETER_AXIS_TITLES.get(kind, ("ParameterA", "ParameterB"))
    for polarity in POLARITY_ORDER:
        polarity_missions = kind_missions[kind_missions["Polarity"] == polarity]
        if polarity_missions.empty:
            continue
        selections = polarity_missions.pivot(index="ParameterB", columns="ParameterA", values="Selected").sort_index().sort_index(axis=1)
        figure = px.imshow(
            selections, color_continuous_scale=SEQUENTIAL_BLUE_SCALE, aspect="auto", origin="lower",
            labels={"x": a_title, "y": b_title, "color": "Selections"},
        )
        figure.update_traces(hovertemplate=f"{a_title} %{{x}}, {b_title} %{{y}}<br>Selections: %{{z}}<extra></extra>")
        style_figure(figure, f"{kind} ({polarity}): selections per parameter pair (gray = not in catalog)")
        figure.update_xaxes(dtick=1)
        figure.update_yaxes(dtick=1, showgrid=False)
        figure.update_layout(plot_bgcolor=MISSING_CELL_COLOR)
        figure.show()


def plot_parameter_distribution(missions, kind):
    # Kindのパラメータ数に応じて、解放済みMissionのパラメータ別選出回数を描く。
    kind_missions = missions[(missions["Kind"] == kind) & (missions["ExposedDraws"] > 0)]
    if kind_missions.empty:
        return
    if PARAMETER_COUNT_BY_KIND.get(kind, 2) == 1:
        plot_single_parameter_distribution(kind_missions, kind)
    else:
        plot_pair_parameter_distribution(kind_missions, kind)


parameterized_kinds = [kind for kind in mission_frequency["Kind"].unique() if PARAMETER_COUNT_BY_KIND.get(kind, 2) > 0]
for kind in sorted(parameterized_kinds, key=lambda name: -mission_frequency.loc[mission_frequency["Kind"] == name, "Selected"].sum()):
    plot_parameter_distribution(mission_frequency, kind)

## 4. 連続出題の偏り

`SequentialFlow` の実行（Session内でSectionが1ずつ進む）でのみ計算します。基準値は「各抽選が全体のKind構成比どおりに独立に選ばれた場合」の一致確率（Σp²）です。実測が基準を大きく上回る場合、同じKindが続きやすいことを示します。同じMissionの直後連続は直前Missionの除外によりほぼ0になるはずです。

In [ ]:
def add_previous_draw_columns(draws):
    # Session内で1つ前の抽選のKind・Mission・極性を列として付ける。
    sequenced = draws.sort_values("DrawIndex").copy()
    grouped = sequenced.groupby("SessionNumber")
    for column in ["SelectedKind", "CatalogIndex", "SelectedPolarity"]:
        sequenced["Previous" + column] = grouped[column].shift()
    return sequenced[sequenced["PreviousSelectedKind"].notna()]


def repeated_within_window(draws, window):
    # 各抽選のMissionが、同じSessionの直前window回以内に出ていたかどうか。
    flags = []
    for _, session in draws.sort_values("DrawIndex").groupby("SessionNumber"):
        history = session["CatalogIndex"].tolist()
        flags.extend(history[index] in history[max(0, index - window):index] for index in range(len(history)))
    return pd.Series(flags, index=draws.sort_values("DrawIndex").index)


def longest_kind_runs(draws):
    # Sessionごとに、同じKindが連続した最長回数を返す。
    def longest_run(kinds):
        run_ids = (kinds != kinds.shift()).cumsum()
        return int(kinds.groupby(run_ids).size().max())

    return draws.sort_values("DrawIndex").groupby("SessionNumber")["SelectedKind"].apply(longest_run)


def summarize_consecutive_repeats(draws, window):
    pairs = add_previous_draw_columns(draws)
    kind_shares = draws["SelectedKind"].value_counts(normalize=True)
    polarity_shares = draws["SelectedPolarity"].value_counts(normalize=True)
    return pd.DataFrame({
        "Observed": [
            (pairs["SelectedKind"] == pairs["PreviousSelectedKind"]).mean(),
            (pairs["SelectedPolarity"] == pairs["PreviousSelectedPolarity"]).mean(),
            (pairs["CatalogIndex"] == pairs["PreviousCatalogIndex"]).mean(),
            repeated_within_window(draws, window).mean(),
        ],
        "IndependentBaseline": [(kind_shares ** 2).sum(), (polarity_shares ** 2).sum(), np.nan, np.nan],
    }, index=["Same kind as previous", "Same polarity as previous", "Same mission as previous", f"Mission seen in previous {window} draws"])


def plot_kind_transitions(draws):
    # 直前Kind → 次Kindの遷移確率（行ごとに正規化）をヒートマップで描く。
    pairs = add_previous_draw_columns(draws)
    kind_order = draws["SelectedKind"].value_counts().index.tolist()
    transitions = pd.crosstab(pairs["PreviousSelectedKind"], pairs["SelectedKind"], normalize="index").reindex(index=kind_order, columns=kind_order, fill_value=0)
    figure = px.imshow(
        transitions, color_continuous_scale=SEQUENTIAL_BLUE_SCALE, aspect="auto",
        labels={"x": "Next kind", "y": "Previous kind", "color": "Probability"},
    )
    figure.update_traces(hovertemplate="%{y} → %{x}: %{z:.1%}<extra></extra>")
    figure.update_coloraxes(colorbar_tickformat=".0%")
    style_figure(figure, "Kind transition probability (row-normalized)", height=max(PLOT_HEIGHT, 30 * len(kind_order))).show()


if IS_SEQUENTIAL_RUN:
    display(summarize_consecutive_repeats(draws, REPEAT_WINDOW_DRAWS).round(4))
    kind_runs = longest_kind_runs(draws)
    display(Markdown(f"**Sessionごとの同一Kind最長連続回数**: 平均 {kind_runs.mean():.2f} / 最大 {kind_runs.max()}"))
    display(kind_runs.value_counts().sort_index().rename_axis("LongestRun").to_frame("Sessions"))
    plot_kind_transitions(draws)
else:
    display(Markdown("この実行は `SequentialFlow` ではないため、連続出題の解析を省略します。"))